# F1 Lap-Level Precipitation

Distributes hourly weather-archive precipitation onto individual lap records so every lap carries a fair share of the rain that fell during that hour.

## Methodology

The Open-Meteo archive delivers precipitation as a single value per calendar hour (mm of rain in that hour).  
For each race the window covers **−2 h → +2 h** relative to race start, giving five hourly buckets:

| Bucket | Weather column | Meaning |
|---|---|---|
| `−2` | `precipitation_hm2` | 2 h before race start's UTC hour |
| `−1` | `precipitation_hm1` | 1 h before race start's UTC hour |
| `0`  | `precipitation_hp0` | UTC hour containing race start |
| `+1` | `precipitation_hp1` | 1 h after race start's UTC hour |
| `+2` | `precipitation_hp2` | 2 h after race start's UTC hour |

Each lap is assigned to the bucket whose wall-clock window contains its `LapStartTimeUTC`.  
The hourly precipitation is then divided equally across **all lap records** (all drivers) that fall in that bucket:

```
Precipitation_Amount  =  precip_mm_in_hour  /  total_laps_in_that_hour
```

**`Cumulative_Precipitation_Amount`** is the running sum of `Precipitation_Amount` for each driver as their laps progress chronologically.

In [7]:
%pip install -q pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)
print("Libraries loaded.")

Libraries loaded.


## Load data

In [9]:
weather = pd.read_csv("../data/f1_race_weather_data.csv")
weather["StartTime"] = pd.to_datetime(weather["StartTime"], utc=True)

precip_cols_available = [c for c in weather.columns if c.startswith("precipitation_h")]
print(f"Weather rows   : {len(weather)}  |  years: {weather['Year'].min()}–{weather['Year'].max()}")
print(f"Precip columns : {precip_cols_available}")

laps = pd.read_pickle("../data/f1_lap_weather_data.pkl")
laps["LapStartTimeUTC"] = pd.to_datetime(laps["LapStartTimeUTC"], utc=True)

print(f"\nLap rows       : {len(laps):,}  |  races: {laps['EventName'].nunique()}")
laps[["Year", "EventName", "Driver", "LapNumber", "LapStartTimeUTC", "Compound"]].head(5)

Weather rows   : 173  |  years: 2018–2025
Precip columns : ['precipitation_hm2', 'precipitation_hm1', 'precipitation_hp0', 'precipitation_hp1', 'precipitation_hp2']

Lap rows       : 188,482  |  races: 36


,Year,EventName,Driver,LapNumber,LapStartTimeUTC,Compound
0,2018,Australian Grand Prix,GAS,1.000000,2018-03-25 05:17:07+00:00,ULTRASOFT
1,2018,Australian Grand Prix,GAS,2.000000,2018-03-25 05:18:53+00:00,ULTRASOFT
2,2018,Australian Grand Prix,GAS,3.000000,2018-03-25 05:20:26+00:00,ULTRASOFT
3,2018,Australian Grand Prix,GAS,4.000000,2018-03-25 05:21:59+00:00,ULTRASOFT
4,2018,Australian Grand Prix,GAS,5.000000,2018-03-25 05:23:31+00:00,ULTRASOFT


## Preparation — hourly precipitation lookup

`race_hour_start` is the UTC hour that contains each race's start time (e.g. a 14:10 UTC start → `race_hour_start = 14:00 UTC`).  
This is the anchor for mapping each lap's `LapStartTimeUTC` to the correct weather bucket.

In [10]:
# Build a long-form table: one row per race × hour offset
PRECIP_HOUR_MAP = {
    -2: "precipitation_hm2",
    -1: "precipitation_hm1",
     0: "precipitation_hp0",
     1: "precipitation_hp1",
     2: "precipitation_hp2",
}

precip_parts = []
for offset, col in PRECIP_HOUR_MAP.items():
    if col not in weather.columns:
        print(f"  Column {col} not found — skipping offset {offset:+d}")
        continue
    part = weather[["Year", "EventName", col]].copy()
    part = part.rename(columns={col: "precip_mm"})
    part["hour_offset"] = offset
    precip_parts.append(part)

precip_long = pd.concat(precip_parts, ignore_index=True)
precip_long["precip_mm"] = precip_long["precip_mm"].fillna(0.0)

print(f"Precipitation lookup: {len(precip_long)} rows")
print(f"Hour offsets covered: {sorted(precip_long['hour_offset'].unique())}")

# Compute the hour anchor from the first actual lap per race, not the scheduled StartTime.
# This correctly handles races with significant delays (red flags before the start, etc.)
# where the first racing lap occurs in a different UTC hour than the scheduled StartTime.
# hour_offset 0 = precipitation_hp0
# hour_offset 1 = precipitation_hp1  ...and so on.
first_lap_anchor = (
    laps.groupby(["Year", "EventName"])["LapStartTimeUTC"]
    .min()
    .dt.floor("h")
    .reset_index(name="race_hour_start")
)

print(f"\nFirst-lap hour anchors computed for {len(first_lap_anchor)} races.")
precip_long[precip_long["precip_mm"] > 0].head(10)

Precipitation lookup: 865 rows
Hour offsets covered: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2)]

First-lap hour anchors computed for 172 races.


,Year,EventName,precip_mm,hour_offset
4,2018,Spanish Grand Prix,0.100000,-2
13,2018,Italian Grand Prix,0.800000,-2
18,2018,Mexican Grand Prix,0.100000,-2
19,2018,Brazilian Grand Prix,0.100000,-2
26,2019,Monaco Grand Prix,0.300000,-2
31,2019,German Grand Prix,0.200000,-2
34,2019,Italian Grand Prix,5.300000,-2
36,2019,Russian Grand Prix,0.200000,-2
38,2019,Mexican Grand Prix,0.100000,-2
44,2020,Hungarian Grand Prix,0.100000,-2


## Assign precipitation to laps

For each lap record:
1. Compute `hour_offset = floor((LapStartTimeUTC − race_hour_start) / 1 h)`
2. Look up `precip_mm` for that race + offset
3. Count all lap records sharing the same race + offset
4. `Precipitation_Amount = precip_mm / laps_in_hour`

In [11]:
# Attach the first-lap hour anchor to every lap record
laps_work = laps.merge(first_lap_anchor, on=["Year", "EventName"], how="left")

# Integer hour offset for each lap relative to its race's first-lap anchor
time_delta = laps_work["LapStartTimeUTC"] - laps_work["race_hour_start"]
raw_offset  = (time_delta.dt.total_seconds() // 3600).astype("Int64")

# Keep only offsets within the weather window (-2 to +2); outside = no data
laps_work["hour_offset"] = raw_offset.where(raw_offset.between(-2, 2), other=pd.NA)

# Join the precipitation amounts
laps_work = laps_work.merge(
    precip_long,
    on=["Year", "EventName", "hour_offset"],
    how="left",
)
laps_work["precip_mm"] = laps_work["precip_mm"].fillna(0.0)

# Count each driver's own laps per race per hour window.
# Using per-driver counts ensures each driver's Precipitation_Amount reflects
# how their hourly rain total is spread across their own laps, not diluted
# by the combined lap counts of all 20 drivers.
driver_lap_counts = (
    laps_work
    .groupby(["Year", "EventName", "Driver", "hour_offset"], dropna=False)
    .size()
    .reset_index(name="driver_laps_in_hour")
)
laps_work = laps_work.merge(
    driver_lap_counts,
    on=["Year", "EventName", "Driver", "hour_offset"],
    how="left",
)

# Distribute hourly precipitation evenly across that driver's laps in the window
laps_work["Precipitation_Amount"] = (
    laps_work["precip_mm"] / laps_work["driver_laps_in_hour"]
).fillna(0.0)

print(f"Laps processed        : {len(laps_work):,}")
print(f"Races covered         : {laps_work['EventName'].nunique()}")
print(f"Laps with rain (>0mm) : {(laps_work['precip_mm'] > 0).sum():,}")

Laps processed        : 188,482
Races covered         : 36
Laps with rain (>0mm) : 37,262


## Cumulative precipitation per driver

After sorting each driver's laps chronologically within a race, the cumulative sum gives the **total rainfall exposure** that driver has experienced up to and including each lap.

In [12]:
laps_work = laps_work.sort_values(
    ["Year", "EventName", "Driver", "LapStartTimeUTC"],
    na_position="last",
).reset_index(drop=True)

laps_work["Cumulative_Precipitation_Amount"] = (
    laps_work
    .groupby(["Year", "EventName", "Driver"])["Precipitation_Amount"]
    .cumsum()
)

print("Cumulative precipitation computed.")
print(f"Max per-driver cumulative : {laps_work['Cumulative_Precipitation_Amount'].max():.6f} mm")
print(f"Max single-lap amount     : {laps_work['Precipitation_Amount'].max():.6f} mm")

Cumulative precipitation computed.
Max per-driver cumulative : 5.100000 mm
Max single-lap amount     : 2.600000 mm


## Preview — sample from the wettest race

In [13]:
# Find the race with the highest single-hour precipitation
wet_race_key = (
    laps_work[laps_work["precip_mm"] > 0]
    .groupby(["Year", "EventName"])["precip_mm"]
    .max()
    .idxmax()
)
yr, ev = wet_race_key
print(f"Wettest race in dataset: {yr} {ev}\n")

preview_cols = [
    "Driver", "LapNumber", "LapStartTimeUTC",
    "hour_offset", "precip_mm", "driver_laps_in_hour",
    "Precipitation_Amount", "Cumulative_Precipitation_Amount",
]

sample = laps_work[(laps_work["Year"] == yr) & (laps_work["EventName"] == ev)]
top_drivers = sample["Driver"].unique()[:3]
(
    sample[sample["Driver"].isin(top_drivers)]
    .groupby("Driver")
    .head(8)
    [preview_cols]
    .sort_values(["Driver", "LapNumber"])
)

Wettest race in dataset: 2019 German Grand Prix



,Driver,LapNumber,LapStartTimeUTC,hour_offset,precip_mm,driver_laps_in_hour,Precipitation_Amount,Cumulative_Precipitation_Amount
33439,ALB,1.000000,2019-07-28 13:49:58+00:00,0,0.100000,6,0.016667,0.016667
33440,ALB,2.000000,2019-07-28 13:51:55+00:00,0,0.100000,6,0.016667,0.033333
33441,ALB,3.000000,2019-07-28 13:53:43+00:00,0,0.100000,6,0.016667,0.050000
33442,ALB,4.000000,2019-07-28 13:55:54+00:00,0,0.100000,6,0.016667,0.066667
33443,ALB,5.000000,2019-07-28 13:57:52+00:00,0,0.100000,6,0.016667,0.083333
33444,ALB,6.000000,2019-07-28 13:59:32+00:00,0,0.100000,6,0.016667,0.100000
33445,ALB,7.000000,2019-07-28 14:01:08+00:00,1,0.100000,36,0.002778,0.102778
33446,ALB,8.000000,2019-07-28 14:02:44+00:00,1,0.100000,36,0.002778,0.105556
33503,BOT,1.000000,2019-07-28 13:49:58+00:00,0,0.100000,6,0.016667,0.016667
33504,BOT,2.000000,2019-07-28 13:51:41+00:00,0,0.100000,6,0.016667,0.033333


## Summary — races with precipitation

In [14]:
# Compute per-race totals (sum unique hourly buckets to avoid double-counting)
race_precip_totals = (
    laps_work[laps_work["hour_offset"].notna()]
    .drop_duplicates(subset=["Year", "EventName", "hour_offset"])
    .groupby(["Year", "EventName"])["precip_mm"]
    .sum()
    .reset_index(name="total_precip_5hr_mm")
)

race_summary = (
    laps_work
    .groupby(["Year", "EventName"])
    .agg(
        total_laps=("LapNumber", "count"),
        max_cumulative=("Cumulative_Precipitation_Amount", "max"),
    )
    .reset_index()
    .merge(race_precip_totals, on=["Year", "EventName"], how="left")
    .query("total_precip_5hr_mm > 0")
    .sort_values("total_precip_5hr_mm", ascending=False)
    .reset_index(drop=True)
)

print(f"Races with any precipitation: {len(race_summary)}")
race_summary

Races with any precipitation: 49


,Year,EventName,total_laps,max_cumulative,total_precip_5hr_mm
0,2024,São Paulo Grand Prix,1134,5.100000,5.100000
1,2019,German Grand Prix,1059,4.400000,4.400000
2,2022,Japanese Grand Prix,507,3.200000,3.200000
3,2018,Spanish Grand Prix,1021,3.000000,3.000000
4,2025,British Grand Prix,825,2.700000,2.700000
5,2024,Canadian Grand Prix,1272,2.500000,2.500000
6,2020,Hungarian Grand Prix,1327,2.300000,2.300000
7,2025,Australian Grand Prix,927,2.000000,2.000000
8,2018,German Grand Prix,1252,1.700000,1.700000
9,2019,Mexican Grand Prix,1370,1.300000,1.300000


## Save combined dataset

Writes `../data/lap_precipitation_data.csv` — one row per lap record, with the two new precipitation columns appended.

In [15]:
OUTPUT_COLS = [
    "Year", "EventName", "Location",
    "Driver", "DriverNumber", "Team",
    "LapNumber", "LapTime",
    "LapStartTimeUTC",
    "Compound", "TyreLife", "FreshTyre",
    "Position", "TrackStatus",
    "hour_offset", "precip_mm", "driver_laps_in_hour",
    "Precipitation_Amount",
    "Cumulative_Precipitation_Amount",
]
output_cols = [c for c in OUTPUT_COLS if c in laps_work.columns]

out_path = "../data/lap_precipitation_data.csv"
laps_work[output_cols].to_csv(out_path, index=False)

print(f"Saved {len(laps_work):,} rows × {len(output_cols)} columns → {out_path}")
laps_work[output_cols].head(5)

Saved 188,482 rows × 19 columns → ../data/lap_precipitation_data.csv


,Year,EventName,Location,Driver,DriverNumber,Team,LapNumber,LapTime,LapStartTimeUTC,Compound,TyreLife,FreshTyre,Position,TrackStatus,hour_offset,precip_mm,driver_laps_in_hour,Precipitation_Amount,Cumulative_Precipitation_Amount
0,2018,Abu Dhabi Grand Prix,Yas Marina,ALO,14,McLaren,1.000000,0 days 00:02:40.620000,2018-11-25 13:43:34+00:00,ULTRASOFT,1.000000,True,14.000000,124.000000,0,0.000000,8,0.000000,0.000000
1,2018,Abu Dhabi Grand Prix,Yas Marina,ALO,14,McLaren,2.000000,0 days 00:02:39.285000,2018-11-25 13:46:15+00:00,ULTRASOFT,2.000000,True,14.000000,4.000000,0,0.000000,8,0.000000,0.000000
2,2018,Abu Dhabi Grand Prix,Yas Marina,ALO,14,McLaren,3.000000,0 days 00:02:35.309000,2018-11-25 13:48:54+00:00,ULTRASOFT,3.000000,True,14.000000,4.000000,0,0.000000,8,0.000000,0.000000
3,2018,Abu Dhabi Grand Prix,Yas Marina,ALO,14,McLaren,4.000000,0 days 00:02:35.918000,2018-11-25 13:51:29+00:00,ULTRASOFT,4.000000,True,14.000000,41.000000,0,0.000000,8,0.000000,0.000000
4,2018,Abu Dhabi Grand Prix,Yas Marina,ALO,14,McLaren,5.000000,0 days 00:01:48.819000,2018-11-25 13:54:05+00:00,ULTRASOFT,5.000000,True,14.000000,1.000000,0,0.000000,8,0.000000,0.000000
